In [ ]:
!pip install duckdb requests pymorphy3

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
import re
import json
import time
import requests
import duckdb

from datetime import datetime
from collections import Counter
import pymorphy3




VK_TOKEN = "e8b64951e8b64951e8b6495121eb892650ee8b6e8b64951810f095b1858c6a8cf76f155"
VK_API_VERSION = "5.199"

SEARCH_QUERY = "искусственный интеллект"
PERIOD = "2025"   # "2025" | "2024-2025" | "2025-01-01:2025-12-31"

MAX_POSTS = 1000
LOAD_COMMENTS = True
COMMENTS_PER_POST_LIMIT = 300

DB_DIR = "/content/drive/MyDrive/vk_project"
DB_PATH = f"{DB_DIR}/vk_parser.duckdb"




os.makedirs(DB_DIR, exist_ok=True)
con = duckdb.connect(DB_PATH)

con.execute("""
CREATE TABLE IF NOT EXISTS vk_users (
    user_id BIGINT PRIMARY KEY,
    vk_user_id BIGINT UNIQUE,
    display_name VARCHAR,
    profile_url VARCHAR
)
""")

con.execute("""
CREATE SEQUENCE IF NOT EXISTS seq_vk_users START 1
""")

con.execute("""
CREATE TABLE IF NOT EXISTS vk_posts (
    post_id BIGINT PRIMARY KEY,
    vk_post_id VARCHAR UNIQUE,
    author_id BIGINT,
    published_at TIMESTAMP,
    text TEXT,
    like_count INTEGER,
    comment_count INTEGER
)
""")

con.execute("""
CREATE SEQUENCE IF NOT EXISTS seq_vk_posts START 1
""")

con.execute("""
CREATE TABLE IF NOT EXISTS vk_comments (
    comment_id BIGINT PRIMARY KEY,
    vk_comment_id VARCHAR UNIQUE,
    post_id BIGINT,
    author_id BIGINT,
    published_at TIMESTAMP,
    text TEXT,
    like_count INTEGER
)
""")

con.execute("""
CREATE SEQUENCE IF NOT EXISTS seq_vk_comments START 1
""")

con.execute("""
CREATE TABLE IF NOT EXISTS nlp_analysis_posts (
    post_an_id BIGINT PRIMARY KEY,
    post_id BIGINT UNIQUE,
    user_post_id BIGINT,
    n_words INTEGER,
    n_inform_word INTEGER,
    sentiment VARCHAR,
    sentiment_score DOUBLE,
    toxicity_score DOUBLE,
    topics TEXT,
    top VARCHAR
)
""")

con.execute("""
CREATE SEQUENCE IF NOT EXISTS seq_nlp_analysis_posts START 1
""")

con.execute("""
CREATE TABLE IF NOT EXISTS nlp_analysis_comments (
    com_an_id BIGINT PRIMARY KEY,
    post_id BIGINT,
    comment_id BIGINT UNIQUE,
    n_words INTEGER,
    n_inform_word INTEGER,
    sentiment VARCHAR,
    sentiment_score DOUBLE,
    toxicity_score DOUBLE,
    topics TEXT
)
""")

con.execute("""
CREATE SEQUENCE IF NOT EXISTS seq_nlp_analysis_comments START 1
""")

print("DuckDB ready:", DB_PATH)




morph = pymorphy3.MorphAnalyzer()

STOPWORDS = {
    "и", "в", "во", "не", "что", "он", "на", "я", "с", "со", "как", "а", "то",
    "все", "она", "так", "его", "но", "да", "ты", "к", "у", "же", "вы", "за",
    "бы", "по", "только", "ее", "мне", "было", "вот", "от", "меня", "еще",
    "нет", "о", "из", "ему", "теперь", "когда", "даже", "ну", "вдруг", "ли",
    "если", "уже", "или", "ни", "быть", "был", "него", "до", "вас", "нибудь",
    "опять", "уж", "вам", "ведь", "там", "потом", "себя", "ничего", "ей",
    "может", "они", "тут", "где", "есть", "надо", "ней", "для", "мы", "тебя",
    "их", "чем", "была", "сам", "чтоб", "без", "будто", "чего", "раз", "тоже",
    "себе", "под", "будет", "ж", "тогда", "кто", "этот", "того", "потому",
    "этого", "какой", "совсем", "ним", "здесь", "этом", "один", "почти",
    "мой", "тем", "чтобы", "нее", "сейчас", "были", "куда", "зачем", "всех",
    "никогда", "можно", "при", "наконец", "два", "об", "другой", "хоть",
    "после", "над", "больше", "тот", "через", "эти", "нас", "про", "всего",
    "них", "какая", "много", "разве", "три", "эту", "моя", "впрочем", "хорошо",
    "свою", "этой", "перед", "иногда", "лучше", "чуть", "том", "нельзя",
    "такой", "им", "более", "всегда", "конечно", "всю", "между"
}

POS_EXCLUDE = {"PREP", "CONJ", "PRCL", "INTJ", "NPRO"}

POSITIVE_WORDS = {
    "хороший", "отличный", "супер", "крутой", "любить", "нравиться", "радость",
    "класс", "прекрасный", "успех", "лучший", "полезный", "добрый"
}
NEGATIVE_WORDS = {
    "плохой", "ужасный", "ненавидеть", "бесить", "отстой", "кошмар", "злой",
    "провал", "тупой", "мерзкий", "отвратительный", "печальный"
}
TOXIC_WORDS = {
    "идиот", "тупой", "дурак", "дебил", "ненавижу", "мразь", "урод", "сволочь"
}


def tokenize(text: str):
    if not text:
        return []
    return re.findall(r"[A-Za-zА-Яа-яЁё0-9\-]+", text.lower())


def normalize_word(word: str):
    parsed = morph.parse(word)[0]
    lemma = parsed.normal_form
    pos = parsed.tag.POS
    return lemma, pos


def analyze_text(text: str):
    tokens = tokenize(text)

    if not tokens:
        return {
            "n_words": 0,
            "n_inform_word": 0,
            "sentiment": "neutral",
            "sentiment_score": 0.0,
            "toxicity_score": 0.0,
            "topics": json.dumps([], ensure_ascii=False),
            "top": None
        }

    informative = []

    for token in tokens:
        lemma, pos = normalize_word(token)
        if lemma not in STOPWORDS and pos not in POS_EXCLUDE and len(lemma) > 2:
            informative.append(lemma)

    counts = Counter(informative)
    top_word = counts.most_common(1)[0][0] if counts else None
    topics = [w for w, _ in counts.most_common(5)]

    pos_cnt = sum(1 for w in informative if w in POSITIVE_WORDS)
    neg_cnt = sum(1 for w in informative if w in NEGATIVE_WORDS)
    toxic_cnt = sum(1 for w in informative if w in TOXIC_WORDS)

    sent_score = (pos_cnt - neg_cnt) / max(len(informative), 1)

    if sent_score > 0.03:
        sentiment = "positive"
    elif sent_score < -0.03:
        sentiment = "negative"
    else:
        sentiment = "neutral"

    toxicity_score = toxic_cnt / max(len(informative), 1)

    return {
        "n_words": len(tokens),
        "n_inform_word": len(informative),
        "sentiment": sentiment,
        "sentiment_score": round(sent_score, 4),
        "toxicity_score": round(toxicity_score, 4),
        "topics": json.dumps(topics, ensure_ascii=False),
        "top": top_word
    }


def parse_period(period_str: str):
    period_str = period_str.strip()

    if re.fullmatch(r"\d{4}", period_str):
        year = int(period_str)
        return (
            datetime(year, 1, 1, 0, 0, 0),
            datetime(year, 12, 31, 23, 59, 59)
        )

    if re.fullmatch(r"\d{4}-\d{4}", period_str):
        start_year, end_year = map(int, period_str.split("-"))
        return (
            datetime(start_year, 1, 1, 0, 0, 0),
            datetime(end_year, 12, 31, 23, 59, 59)
        )

    if ":" in period_str:
        left, right = period_str.split(":")
        return (
            datetime.strptime(left.strip(), "%Y-%m-%d"),
            datetime.strptime(right.strip(), "%Y-%m-%d").replace(hour=23, minute=59, second=59)
        )

    raise ValueError("Неверный формат PERIOD")


def vk_method(method: str, params: dict):
    url = f"https://api.vk.com/method/{method}"

    params = params.copy()
    params["access_token"] = VK_TOKEN
    params["v"] = VK_API_VERSION

    response = requests.get(url, params=params, timeout=30)
    data = response.json()

    if "error" in data:
        raise RuntimeError(f"VK API error in {method}: {data['error']}")

    return data["response"]


def build_profile_url(vk_id: int, screen_name: str = None):
    if screen_name:
        return f"https://vk.com/{screen_name}"
    if vk_id < 0:
        return f"https://vk.com/club{abs(vk_id)}"
    return f"https://vk.com/id{vk_id}"




def get_user_by_vk_id(vk_user_id: int):
    row = con.execute("""
        SELECT user_id, vk_user_id, display_name, profile_url
        FROM vk_users
        WHERE vk_user_id = ?
    """, [vk_user_id]).fetchone()
    return row


def create_user(vk_user_id: int, display_name: str, profile_url: str):
    user_id = con.execute("SELECT nextval('seq_vk_users')").fetchone()[0]

    con.execute("""
        INSERT INTO vk_users (user_id, vk_user_id, display_name, profile_url)
        VALUES (?, ?, ?, ?)
    """, [user_id, vk_user_id, display_name, profile_url])

    return user_id


def get_or_create_user(vk_user_id: int, display_name: str, profile_url: str):
    existing = get_user_by_vk_id(vk_user_id)

    if existing:
        user_id = existing[0]
        con.execute("""
            UPDATE vk_users
            SET display_name = ?, profile_url = ?
            WHERE user_id = ?
        """, [display_name, profile_url, user_id])
        return user_id

    return create_user(vk_user_id, display_name, profile_url)


def upsert_users_from_extended(profiles, groups):
    vk_to_local = {}

    for p in profiles or []:
        vk_id = int(p["id"])
        display_name = f"{p.get('first_name', '')} {p.get('last_name', '')}".strip()
        if not display_name:
            display_name = f"id{vk_id}"
        profile_url = build_profile_url(vk_id, p.get("screen_name"))
        local_id = get_or_create_user(vk_id, display_name, profile_url)
        vk_to_local[vk_id] = local_id

    for g in groups or []:
        vk_id = -int(g["id"])
        display_name = g.get("name", f"club{abs(vk_id)}")
        profile_url = build_profile_url(vk_id, g.get("screen_name"))
        local_id = get_or_create_user(vk_id, display_name, profile_url)
        vk_to_local[vk_id] = local_id

    return vk_to_local


def get_post_by_vk_post_id(vk_post_id: str):
    row = con.execute("""
        SELECT post_id, vk_post_id, author_id, published_at, text, like_count, comment_count
        FROM vk_posts
        WHERE vk_post_id = ?
    """, [vk_post_id]).fetchone()
    return row


def insert_post(vk_post_id, author_id, published_at, text, like_count, comment_count):
    post_id = con.execute("SELECT nextval('seq_vk_posts')").fetchone()[0]

    con.execute("""
        INSERT INTO vk_posts (post_id, vk_post_id, author_id, published_at, text, like_count, comment_count)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, [post_id, vk_post_id, author_id, published_at, text, like_count, comment_count])

    return post_id


def insert_post_analysis(post_id, user_post_id, nlp):
    exists = con.execute("""
        SELECT 1 FROM nlp_analysis_posts WHERE post_id = ?
    """, [post_id]).fetchone()

    if exists:
        return

    post_an_id = con.execute("SELECT nextval('seq_nlp_analysis_posts')").fetchone()[0]

    con.execute("""
        INSERT INTO nlp_analysis_posts (
            post_an_id, post_id, user_post_id, n_words, n_inform_word,
            sentiment, sentiment_score, toxicity_score, topics, top
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, [
        post_an_id,
        post_id,
        user_post_id,
        nlp["n_words"],
        nlp["n_inform_word"],
        nlp["sentiment"],
        nlp["sentiment_score"],
        nlp["toxicity_score"],
        nlp["topics"],
        nlp["top"]
    ])


def get_comment_by_vk_comment_id(vk_comment_id: str):
    row = con.execute("""
        SELECT comment_id
        FROM vk_comments
        WHERE vk_comment_id = ?
    """, [vk_comment_id]).fetchone()
    return row


def insert_comment(vk_comment_id, post_id, author_id, published_at, text, like_count):
    comment_id = con.execute("SELECT nextval('seq_vk_comments')").fetchone()[0]

    con.execute("""
        INSERT INTO vk_comments (comment_id, vk_comment_id, post_id, author_id, published_at, text, like_count)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, [comment_id, vk_comment_id, post_id, author_id, published_at, text, like_count])

    return comment_id


def insert_comment_analysis(post_id, comment_id, nlp):
    exists = con.execute("""
        SELECT 1 FROM nlp_analysis_comments WHERE comment_id = ?
    """, [comment_id]).fetchone()

    if exists:
        return

    com_an_id = con.execute("SELECT nextval('seq_nlp_analysis_comments')").fetchone()[0]

    con.execute("""
        INSERT INTO nlp_analysis_comments (
            com_an_id, post_id, comment_id, n_words, n_inform_word,
            sentiment, sentiment_score, toxicity_score, topics
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, [
        com_an_id,
        post_id,
        comment_id,
        nlp["n_words"],
        nlp["n_inform_word"],
        nlp["sentiment"],
        nlp["sentiment_score"],
        nlp["toxicity_score"],
        nlp["topics"]
    ])




def search_posts_global(query: str, dt_from: datetime, dt_to: datetime, max_posts: int = 1000):
    collected_post_ids = []
    next_from = None

    while True:
        remaining = max_posts - len(collected_post_ids)
        if remaining <= 0:
            break

        batch_size = min(200, remaining)

        params = {
            "q": query,
            "extended": 1,
            "count": batch_size,
            "start_time": int(dt_from.timestamp()),
            "end_time": int(dt_to.timestamp()),
            "fields": "screen_name"
        }

        if next_from:
            params["start_from"] = next_from

        resp = vk_method("newsfeed.search", params)

        items = resp.get("items", [])
        profiles = resp.get("profiles", [])
        groups = resp.get("groups", [])
        next_from = resp.get("next_from")

        vk_to_local = upsert_users_from_extended(profiles, groups)

        if not items:
            break

        for item in items:
            owner_id = int(item["owner_id"])
            vk_post_id = f"{owner_id}_{item['id']}"

            existing_post = get_post_by_vk_post_id(vk_post_id)
            if existing_post:
                collected_post_ids.append(existing_post[0])
                continue

            author_local_id = vk_to_local.get(owner_id)
            if not author_local_id:
                author_local_id = get_or_create_user(
                    owner_id,
                    f"vk_{owner_id}",
                    build_profile_url(owner_id)
                )

            published_at = datetime.fromtimestamp(item["date"])
            text = item.get("text", "")
            like_count = item.get("likes", {}).get("count", 0)
            comment_count = item.get("comments", {}).get("count", 0)

            post_id = insert_post(
                vk_post_id=vk_post_id,
                author_id=author_local_id,
                published_at=published_at,
                text=text,
                like_count=like_count,
                comment_count=comment_count
            )

            nlp = analyze_text(text)
            insert_post_analysis(post_id, author_local_id, nlp)

            collected_post_ids.append(post_id)

        if not next_from:
            break

        time.sleep(0.35)

    return collected_post_ids




def load_comments_for_post(post_id: int):
    row = con.execute("""
        SELECT vk_post_id
        FROM vk_posts
        WHERE post_id = ?
    """, [post_id]).fetchone()

    if not row:
        return

    vk_post_id = row[0]
    owner_id_str, post_vk_id_str = vk_post_id.split("_")
    owner_id = int(owner_id_str)
    post_vk_id = int(post_vk_id_str)

    offset = 0
    loaded = 0

    while loaded < COMMENTS_PER_POST_LIMIT:
        batch_size = min(100, COMMENTS_PER_POST_LIMIT - loaded)

        try:
            resp = vk_method("wall.getComments", {
                "owner_id": owner_id,
                "post_id": post_vk_id,
                "need_likes": 1,
                "sort": "asc",
                "offset": offset,
                "count": batch_size,
                "extended": 1,
                "fields": "screen_name"
            })
        except Exception as e:
            print(f"[WARN] comments not loaded for {vk_post_id}: {e}")
            break

        items = resp.get("items", [])
        profiles = resp.get("profiles", [])
        groups = resp.get("groups", [])

        if not items:
            break

        vk_to_local = upsert_users_from_extended(profiles, groups)

        for item in items:
            comment_vk_id = f"{owner_id}_{post_vk_id}_{item['id']}"

            if get_comment_by_vk_comment_id(comment_vk_id):
                continue

            from_id = int(item.get("from_id", 0))
            author_local_id = vk_to_local.get(from_id)

            if not author_local_id:
                author_local_id = get_or_create_user(
                    from_id,
                    f"vk_{from_id}",
                    build_profile_url(from_id)
                )

            text = item.get("text", "")
            published_at = datetime.fromtimestamp(item["date"])
            like_count = item.get("likes", {}).get("count", 0)

            comment_id = insert_comment(
                vk_comment_id=comment_vk_id,
                post_id=post_id,
                author_id=author_local_id,
                published_at=published_at,
                text=text,
                like_count=like_count
            )

            nlp = analyze_text(text)
            insert_comment_analysis(post_id, comment_id, nlp)

        loaded += len(items)
        offset += len(items)

        if len(items) < batch_size:
            break

        time.sleep(0.35)




def main():
    dt_from, dt_to = parse_period(PERIOD)

    print(f"Поиск постов: {SEARCH_QUERY}")
    print(f"Период: {dt_from} .. {dt_to}")

    post_ids = search_posts_global(
        query=SEARCH_QUERY,
        dt_from=dt_from,
        dt_to=dt_to,
        max_posts=MAX_POSTS
    )

    print("Сохранено постов:", len(post_ids))

    if LOAD_COMMENTS:
        for i, post_id in enumerate(post_ids, start=1):
            print(f"[{i}/{len(post_ids)}] Загружаю комментарии для post_id={post_id}")
            load_comments_for_post(post_id)

    print("Готово.")
    print("База:", DB_PATH)


main()

In [ ]:
con.execute("SELECT * FROM vk_users LIMIT 5").fetchdf()

In [ ]:
con.execute("SELECT * FROM vk_users LIMIT 5").fetchdf()

In [ ]:
con.execute("SELECT * FROM vk_posts LIMIT 5").fetchdf()

In [ ]:
con.execute("SELECT * FROM vk_comments LIMIT 5").fetchdf()

In [ ]:
con.execute("SELECT * FROM nlp_analysis_posts LIMIT 20").fetchdf()

## импорт таблиц в csv если надо

In [ ]:
con.execute("COPY vk_users TO '/content/drive/MyDrive/vk_project/vk_users.csv' (HEADER, DELIMITER ',')")
con.execute("COPY vk_posts TO '/content/drive/MyDrive/vk_project/vk_posts.csv' (HEADER, DELIMITER ',')")
con.execute("COPY vk_comments TO '/content/drive/MyDrive/vk_project/vk_comments.csv' (HEADER, DELIMITER ',')")
con.execute("COPY nlp_analysis_posts TO '/content/drive/MyDrive/vk_project/nlp_analysis_posts.csv' (HEADER, DELIMITER ',')")
con.execute("COPY nlp_analysis_comments TO '/content/drive/MyDrive/vk_project/nlp_analysis_comments.csv' (HEADER, DELIMITER ',')")

In [3]:
!pip install leafmap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 666.8/666.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.2 MB/s eta 0:00:00
  Attempting uninstall: duckdb
    Found existing installation: duckdb 1.3.2
    Uninstalling duckdb-1.3.2:
      Successfully uninstalled duckdb-1.3.2


In [8]:
import duckdb

conn = duckdb.connect("/content/string.duckdb")

# посмотреть таблицы
print(conn.execute("SHOW TABLES").fetchall())

# посмотреть данные
df = conn.execute("SELECT * FROM vk_posts LIMIT 10").df()
print(df)

[('nlp_analysis_comments',), ('nlp_analysis_posts',), ('vk_comments',), ('vk_posts',), ('vk_users',)]
   post_id         vk_post_id  author_id        published_at  \
0        1  -160107791_237549         36 2026-03-24 13:26:48   
1        2     789869204_8674          2 2026-03-24 13:03:12   
2        3   -196127788_28288         37 2026-03-24 12:15:00   
3        4   -190209548_14464         38 2026-03-24 12:00:09   
4        5    -193313947_7976         39 2026-03-24 11:58:27   
5        6       -236908396_4         40 2026-03-24 11:22:14   
6        7   -212485639_11016         41 2026-03-24 11:13:00   
7        8   -199534700_31742         42 2026-03-24 11:02:18   
8        9    -193313947_7974         39 2026-03-24 11:01:03   
9       10   -187896399_48452         43 2026-03-24 10:37:54   

                                                text  like_count  \
0  ★★★ΦИЛЬΜ ДНЯ ★★★ \n☾◐ Нa нoчь глядя... ◑☽ \nСт...           0   
1  Таможенный сертификат Противоснарядное энергоп...     